# GraphRAG-Bench (medical) + OSU HippoRAG2 (Ollama)

- Dataset: `GraphRAG-Benchmark/Datasets/Corpus/medical.parquet`
- Engine: `OSU-NLP-Group/HippoRAG` (minor Colab patches for optional deps) + Ollama (`llama3.1:8b`)
- Mode: FULL evaluation (`SAMPLE_N=None`)

Ipucu: `SAMPLE_N=None` => full evaluation (all questions), `SAMPLE_N=10` => fast debug.
Not: `pip install -r requirements.txt` KULLANMA; paketler tek tek kuruluyor.


In [ ]:
import os
from pathlib import Path

MODEL_NAME = "gpt-4o-mini"
OLLAMA_BASE = "http://localhost:11434"
OPENAI_COMPAT_BASE = "https://api.openai.com/v1"  # Ollama OpenAI-compatible endpoint

# Embedding for HippoRAG2 internal retrieval (uses SentenceTransformers via "Transformers/...")
EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"

# Embedding for GraphRAG-Benchmark evaluation (HF BGE embeddings)
EVAL_EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"

SAMPLE_N = 128  # None => full evaluation (all questions)
FORCE_INDEX_FROM_SCRATCH = "true"  # set to "false" to reuse previous index
FORCE_OPENIE_FROM_SCRATCH = "true"  # set to "false" to reuse previous OpenIE
RUN_TAG = None  # auto-filled later

print({
  "MODEL_NAME": MODEL_NAME,
  "EMBED_MODEL_ID": EMBED_MODEL_ID,
  "EVAL_EMBED_MODEL_ID": EVAL_EMBED_MODEL_ID,
  "SAMPLE_N": ("ALL" if SAMPLE_N is None else SAMPLE_N),
})

{'MODEL_NAME': 'gpt-4o-mini', 'EMBED_MODEL_ID': 'BAAI/bge-base-en-v1.5', 'EVAL_EMBED_MODEL_ID': 'BAAI/bge-large-en-v1.5', 'SAMPLE_N': 128}


In [ ]:
# GPU check (Colab: Runtime > Change runtime type > GPU)
!nvidia-smi || true
import importlib.util
if importlib.util.find_spec("torch") is not None:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
else:
    print("torch not found (Colab GPU runtime normally has it).")


Tue Apr 14 15:07:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Clone repos
!git clone https://github.com/GraphRAG-Bench/GraphRAG-Benchmark.git /content/GraphRAG-Benchmark 2>/dev/null || echo "Repo zaten mevcut"
!git clone https://github.com/OSU-NLP-Group/HippoRAG.git /content/HippoRAG 2>/dev/null || echo "Repo zaten mevcut"

print("Dataset hazir!")
!ls /content/GraphRAG-Benchmark/Datasets/Corpus/ | head

Repo zaten mevcut
Repo zaten mevcut
Dataset hazir!
medical.json
medical.parquet
novel.json
novel.parquet


In [ ]:
# Python deps (tek tek, requirements.txt YOK)
# GraphRAG-Benchmark eval deps
!pip -q install \
  datasets==3.3.2 \
  langchain==0.3.26 \
  langchain-core==0.3.69 \
  langchain-openai==0.3.28 \
  langchain-ollama \
  langchain-community==0.3.26 \
  ragas==0.2.15 \
  rouge_score==0.1.2 \
  json5 \
  json_repair \
  python-dotenv \
  aiohttp

# HippoRAG deps (minimum practical set)
!pip -q install \
  openai==1.91.0 \
  litellm==1.73.1 \
  transformers==4.56.2 \
  tiktoken==0.7.0 \
  protobuf \
  sentencepiece \
  pydantic==2.10.4 \
  tenacity==8.5.0 \
  igraph==0.11.9 \
  texttable==1.7.0 \
  networkx==3.4.2 \
  sentence-transformers==4.0.1 \
  gritlm==1.0.2 \
  boto3 \
  pandas \
  pyarrow \
  scipy \
  einops \
  tqdm \
  filelock \
  packaging \
  httpx \
  requests

# Install HippoRAG editable (deps already installed above)
!pip -q install -e /content/HippoRAG --no-deps


  Preparing metadata (setup.py) ... done


In [ ]:
# Patch HippoRAG import guards (avoid hard deps on vllm/outlines for online mode)
from pathlib import Path
import re

hippo_path = Path("/content/HippoRAG/src/hipporag/HippoRAG.py")
s = hippo_path.read_text(encoding="utf-8")

# Replace unconditional offline OpenIE imports with try/except guards
patterns = [
    (r"^from \.information_extraction\.openie_vllm_offline import VLLMOfflineOpenIE\s*$",
     "try:\n    from .information_extraction.openie_vllm_offline import VLLMOfflineOpenIE\nexcept Exception:\n    VLLMOfflineOpenIE = None"),
    (r"^from \.information_extraction\.openie_transformers_offline import TransformersOfflineOpenIE\s*$",
     "try:\n    from .information_extraction.openie_transformers_offline import TransformersOfflineOpenIE\nexcept Exception:\n    TransformersOfflineOpenIE = None"),
]

for pat, repl in patterns:
    s = re.sub(pat, repl, s, flags=re.MULTILINE)

# Add runtime checks when offline modes are requested
if "elif self.global_config.openie_mode == 'offline':" in s and "VLLMOfflineOpenIE is None" not in s:
    s = s.replace(
        "elif self.global_config.openie_mode == 'offline':\n            self.openie = VLLMOfflineOpenIE(self.global_config)",
        "elif self.global_config.openie_mode == 'offline':\n            if VLLMOfflineOpenIE is None:\n                raise RuntimeError(\'openie_mode=offline requires vllm extras which are not installed\')\n            self.openie = VLLMOfflineOpenIE(self.global_config)"
    )
if "elif self.global_config.openie_mode ==  'Transformers-offline':" in s and "TransformersOfflineOpenIE is None" not in s:
    s = s.replace(
        "elif self.global_config.openie_mode ==  'Transformers-offline':\n            self.openie = TransformersOfflineOpenIE(self.global_config)",
        "elif self.global_config.openie_mode ==  'Transformers-offline':\n            if TransformersOfflineOpenIE is None:\n                raise RuntimeError(\'openie_mode=Transformers-offline requires optional deps which are not installed\')\n            self.openie = TransformersOfflineOpenIE(self.global_config)"
    )

hippo_path.write_text(s, encoding="utf-8")
print("Patched:", hippo_path)


Patched: /content/HippoRAG/src/hipporag/HippoRAG.py


In [ ]:
# Patch HippoRAG optional embedding imports (avoid requiring gritlm/cohere/vllm unless used)
from pathlib import Path

embed_init = Path("/content/HippoRAG/src/hipporag/embedding_model/__init__.py")
embed_init.write_text("""from .base import EmbeddingConfig, BaseEmbeddingModel

from ..utils.logging_utils import get_logger

logger = get_logger(__name__)

# Optional embedding backends: guard imports to avoid hard dependencies
try:
    from .Contriever import ContrieverModel
except Exception as e:
    ContrieverModel = None
    logger.warning(f"ContrieverModel unavailable: {e}")

try:
    from .GritLM import GritLMEmbeddingModel
except Exception as e:
    GritLMEmbeddingModel = None
    logger.warning(f"GritLMEmbeddingModel unavailable: {e}")

try:
    from .NVEmbedV2 import NVEmbedV2EmbeddingModel
except Exception as e:
    NVEmbedV2EmbeddingModel = None
    logger.warning(f"NVEmbedV2EmbeddingModel unavailable: {e}")

try:
    from .OpenAI import OpenAIEmbeddingModel
except Exception as e:
    OpenAIEmbeddingModel = None
    logger.warning(f"OpenAIEmbeddingModel unavailable: {e}")

try:
    from .Cohere import CohereEmbeddingModel
except Exception as e:
    CohereEmbeddingModel = None
    logger.warning(f"CohereEmbeddingModel unavailable: {e}")

try:
    from .Transformers import TransformersEmbeddingModel
except Exception as e:
    TransformersEmbeddingModel = None
    logger.warning(f"TransformersEmbeddingModel unavailable: {e}")

try:
    from .VLLM import VLLMEmbeddingModel
except Exception as e:
    VLLMEmbeddingModel = None
    logger.warning(f"VLLMEmbeddingModel unavailable: {e}")


def _require(cls, what: str):
    if cls is None:
        raise RuntimeError(f"Embedding backend not available for {what}. Install its dependency or choose another embedding_model_name.")
    return cls


def _get_embedding_model_class(embedding_model_name: str = "nvidia/NV-Embed-v2"):
    if "GritLM" in embedding_model_name:
        return _require(GritLMEmbeddingModel, "GritLM")
    elif "NV-Embed-v2" in embedding_model_name:
        return _require(NVEmbedV2EmbeddingModel, "NV-Embed-v2")
    elif "contriever" in embedding_model_name:
        return _require(ContrieverModel, "contriever")
    elif "text-embedding" in embedding_model_name:
        return _require(OpenAIEmbeddingModel, "OpenAI text-embedding")
    elif "cohere" in embedding_model_name:
        return _require(CohereEmbeddingModel, "cohere")
    elif embedding_model_name.startswith("Transformers/"):
        return _require(TransformersEmbeddingModel, "Transformers/")
    elif embedding_model_name.startswith("VLLM/"):
        return _require(VLLMEmbeddingModel, "VLLM/")
    assert False, f"Unknown embedding model name: {embedding_model_name}"


__all__ = [
    "EmbeddingConfig",
    "BaseEmbeddingModel",
    "_get_embedding_model_class",
]
""", encoding="utf-8")
print("Patched:", embed_init)


Patched: /content/HippoRAG/src/hipporag/embedding_model/__init__.py


In [ ]:
# Dependency sanity check + auto-install igraph if missing
import sys, subprocess
print("python:", sys.version)
try:
    import igraph  # provided by pip package `igraph`
except ModuleNotFoundError:
    print("igraph not found; installing igraph wheel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "igraph==0.11.9", "texttable==1.7.0"])
    import igraph
print("igraph:", igraph.__version__)
from igraph import Graph
print("igraph Graph import OK")


python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
igraph: 0.11.9
igraph Graph import OK


In [ ]:
# Transformers stack repair (Colab / py3.12)
# Fixes errors like:
# - ModuleNotFoundError: transformers.modeling_layers
# - cannot import name check_torch_load_is_safe from transformers.utils
from pathlib import Path
import importlib
import site
import shutil
import subprocess
import sys

TARGET_TRANSFORMERS = "4.56.2"
TARGET_SENTENCE_TRANSFORMERS = "4.0.1"

PURGE_PREFIXES = ("transformers", "tokenizers", "sentence_transformers")


def purge_loaded_modules():
    for name in list(sys.modules.keys()):
        if any(name == p or name.startswith(p + ".") for p in PURGE_PREFIXES):
            del sys.modules[name]


def pip(*args):
    cmd = [sys.executable, "-m", "pip", *args]
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


def nuke_leftovers():
    roots = set(site.getsitepackages())
    try:
        roots.add(site.getusersitepackages())
    except Exception:
        pass

    for sp in sorted(roots):
        sp = Path(sp)
        for name in ["transformers", "tokenizers", "sentence_transformers"]:
            p = sp / name
            if p.exists():
                print("Removing", p)
                shutil.rmtree(p, ignore_errors=True)

        for pattern in [
            "transformers-*.dist-info",
            "tokenizers-*.dist-info",
            "sentence_transformers-*.dist-info",
        ]:
            for p in sp.glob(pattern):
                print("Removing", p)
                shutil.rmtree(p, ignore_errors=True)


def sanity_check():
    purge_loaded_modules()
    importlib.invalidate_caches()

    import transformers

    print("transformers:", transformers.__version__, transformers.__file__)

    # These are the two imports that kept failing in this runtime.
    import transformers.modeling_layers  # noqa: F401
    from transformers.utils import check_torch_load_is_safe  # noqa: F401

    from transformers import Trainer  # noqa: F401
    print("Trainer import OK")

    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__, sentence_transformers.__file__)


try:
    sanity_check()
except Exception as e:
    print("Transformers stack broken; repairing...\n", repr(e))

    # Uninstall (ignore failures)
    subprocess.call(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "transformers",
            "tokenizers",
            "sentence-transformers",
        ]
    )

    purge_loaded_modules()
    nuke_leftovers()

    # Reinstall cleanly
    pip(
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--force-reinstall",
        f"transformers=={TARGET_TRANSFORMERS}",
    )
    pip(
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--force-reinstall",
        f"sentence-transformers=={TARGET_SENTENCE_TRANSFORMERS}",
    )

    sanity_check()

    print("\nIf you still see mixed/partial imports after this, do: Runtime > Restart runtime, then rerun this cell.")


transformers: 4.56.2 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Trainer import OK
sentence-transformers: 4.0.1 /usr/local/lib/python3.12/dist-packages/sentence_transformers/__init__.py


In [ ]:
# Install + start Ollama
!apt-get update -y >/dev/null
!apt-get install -y zstd zip >/dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# Start server in background
!pkill -f "ollama serve" >/dev/null 2>&1 || true
!nohup ollama serve > /content/ollama.log 2>&1 &

# Wait a bit
import time
time.sleep(2)

# Pull model
!ollama pull "$MODEL_NAME"

# Quick health check
!ollama list | head

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
^C

Error: pull model manifest: file does not exist
NAME    ID    SIZE    MODIFIED 


In [ ]:
# Ollama GPU warmup + quick OpenAI-compat check
import time
print("Ollama base:", OLLAMA_BASE)
print("OpenAI compat:", OPENAI_COMPAT_BASE)
!curl -s "$OPENAI_COMPAT_BASE/models" | head -c 400 || true
print("\nWarming up model...")
!ollama run "$MODEL_NAME" "Say: warmup" | head -n 5
time.sleep(1)
print("\nOllama processes (if any):")
!ollama ps || true
print("\nGPU processes (look for ollama):")
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv || true


Ollama base: http://localhost:11434
OpenAI compat: https://api.openai.com/v1
{
  "error": {
    "message": "Missing bearer authentication in header",
    "type": "invalid_request_error",
    "param": null,
    "code": null
  }
}
Warming up model...

Error: pull model manifest: file does not exist

Ollama processes (if any):
NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 

GPU processes (look for ollama):
pid, process_name, used_gpu_memory [MiB]


In [ ]:
# API keys (OpenAI / GraphRAG-Benchmark Evaluation)
import os

# 1) Put your real OpenAI key here:
os.environ["OPENAI_API_KEY"] = "YourKey"

# 2) GraphRAG-Benchmark Evaluation scripts expect LLM_API_KEY
os.environ["LLM_API_KEY"] = os.environ["OPENAI_API_KEY"]

# (Optional) keep local calls stable if you still use localhost anywhere
for k in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    os.environ.pop(k, None)
os.environ["NO_PROXY"] = "127.0.0.1,localhost"
os.environ["no_proxy"] = "127.0.0.1,localhost"

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))
print("LLM_API_KEY set:", bool(os.environ.get("LLM_API_KEY")))


OPENAI_API_KEY set: True
LLM_API_KEY set: True


In [ ]:
# Patch GraphRAG-Benchmark Examples/run_hipporag2.py for Colab + OSU HippoRAG
# - Remove fixed CUDA_VISIBLE_DEVICES=5
# - Make rerank_dspy_file_path robust via installed hipporag package path
# - Use SentenceTransformers embedding inside HippoRAG by prefixing "Transformers/"
# - Keep tokenizer splitting working by stripping "Transformers/" prefix

import re

MAX_NEW_TOKENS = 256  # cap generation; prevents hanging on long answers

from pathlib import Path
p = Path("/content/GraphRAG-Benchmark/Examples/run_hipporag2.py")
src = p.read_text(encoding="utf-8")

# 1) CUDA device line
src = src.replace('os.environ["CUDA_VISIBLE_DEVICES"] = "5"', '# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Colab')

# 2) Tokenizer load: strip Transformers/ prefix if present
src = src.replace(
    'tokenizer = AutoTokenizer.from_pretrained(embed_model_path)',
    'tokenizer = AutoTokenizer.from_pretrained(embed_model_path.replace("Transformers/", ""))'
)

# 3) Inject robust rerank path + embedding prefixing
needle = '    # Configure HippoRAG\n    config = BaseConfig('
if needle not in src:
    raise RuntimeError("Patch failed: expected marker not found")

inject = (
    '    # Resolve DSPy rerank prompt path from installed hipporag package\n'
    '    try:\n'
    '        import hipporag as _hipporag_pkg\n'
    '        from pathlib import Path as _Path\n'
    '        _rerank_path = _Path(_hipporag_pkg.__file__).resolve().parent / "prompts/dspy_prompts/filter_llama3.3-70B-Instruct.json"\n'
    '        rerank_path = str(_rerank_path) if _rerank_path.exists() else None\n'
    '    except Exception:\n'
    '        rerank_path = None\n\n'
    '    # Use SentenceTransformers embedding inside HippoRAG without modifying OSU engine\n'
    '    # Accept both raw HF ids and already-prefixed values\n'
    '    hipporag_embedding_name = embed_model_path if embed_model_path.startswith("Transformers/") else f"Transformers/{embed_model_path}"\n\n'
)

src = src.replace(needle, inject + needle)

# 4) Swap config fields
src = src.replace('embedding_model_name=embed_model_path,', 'embedding_model_name=hipporag_embedding_name,')
src = src.replace(
    'rerank_dspy_file_path="hipporag/prompts/dspy_prompts/filter_llama3.3-70B-Instruct.json",',
    'rerank_dspy_file_path=rerank_path,'
)

src = src.replace("max_new_tokens=None,", f"max_new_tokens={MAX_NEW_TOKENS},")

p.write_text(src, encoding="utf-8")
print("Patched:", p)

Patched: /content/GraphRAG-Benchmark/Examples/run_hipporag2.py


In [ ]:
# Enable fast reruns (force flags) by patching Examples/run_hipporag2.py
# Fixes: NameError: name 'args' is not defined (BaseConfig is built inside process_corpus)
from pathlib import Path
import re

p = Path("/content/GraphRAG-Benchmark/Examples/run_hipporag2.py")
s = p.read_text(encoding="utf-8")

# 1) Add argparse flags if missing (insert right after the --sample add_argument block)
if "--force_index_from_scratch" not in s:
    m = re.search(r'(parser\.add_argument\(\s*"--sample"[\s\S]*?\)\s*)\n', s)
    if not m:
        raise RuntimeError("Could not locate --sample argument in run_hipporag2.py")
    insert = (
        "    parser.add_argument(\"--force_index_from_scratch\", type=str, default=\"true\", help=\"true/false\")\n"
        "    parser.add_argument(\"--force_openie_from_scratch\", type=str, default=\"true\", help=\"true/false\")\n"
    )
    s = s[: m.end()] + insert + s[m.end() :]

# 2) Thread flags into process_corpus signature (stable anchor block)
sig_block_old = (
    "    questions: Dict[str, List[dict]],\n"
    "    sample: int\n"
    "):"
)

sig_block_new = (
    "    questions: Dict[str, List[dict]],\n"
    "    sample: int,\n"
    "    force_index_from_scratch: bool,\n"
    "    force_openie_from_scratch: bool\n"
    "):"
)

if sig_block_old in s and "force_index_from_scratch: bool" not in s:
    s = s.replace(sig_block_old, sig_block_new)

# 3) Use passed flags inside BaseConfig
s = s.replace("force_index_from_scratch=True,", "force_index_from_scratch=force_index_from_scratch,")
s = s.replace("force_openie_from_scratch=True,", "force_openie_from_scratch=force_openie_from_scratch,")

# 4) Pass flags from args into process_corpus call (asyncio.to_thread)
call_block_old = (
    "                grouped_questions,\n"
    "                args.sample,\n"
    "            ))"
)

call_block_new = (
    "                grouped_questions,\n"
    "                args.sample,\n"
    "                string_to_bool(args.force_index_from_scratch),\n"
    "                string_to_bool(args.force_openie_from_scratch),\n"
    "            ))"
)

if call_block_old in s and "args.force_index_from_scratch" not in s:
    s = s.replace(call_block_old, call_block_new)

p.write_text(s, encoding="utf-8")
print("Patched force flags in:", p)


Patched force flags in: /content/GraphRAG-Benchmark/Examples/run_hipporag2.py


In [ ]:
# Run HippoRAG2 on GraphRAG-Bench medical subset (FULL eval if SAMPLE_N=None) with logging
from datetime import datetime
import os, sys, subprocess

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"/content/GraphRAG-Benchmark/results/runs/{RUN_TAG}"
LOG_DIR = f"{RUN_DIR}/logs"
WORKSPACE_DIR = "/content/workspaces/hipporag2_workspace"
os.makedirs(LOG_DIR, exist_ok=True)

print("RUN_TAG:", RUN_TAG)
print("RUN_DIR:", RUN_DIR)
print("WORKSPACE_DIR:", WORKSPACE_DIR)
print("SAMPLE_N:", "ALL" if SAMPLE_N is None else SAMPLE_N)
print("FORCE_INDEX_FROM_SCRATCH:", FORCE_INDEX_FROM_SCRATCH)
print("FORCE_OPENIE_FROM_SCRATCH:", FORCE_OPENIE_FROM_SCRATCH)

RUN_LOG = f"{LOG_DIR}/run_hipporag2_medical.log"
print("Log:", RUN_LOG)

cmd = [
    sys.executable, "Examples/run_hipporag2.py",
    "--subset", "medical",
    "--mode", "API",
    "--base_dir", WORKSPACE_DIR,
    "--model_name", MODEL_NAME,
    "--embed_model_path", EMBED_MODEL_ID,
    "--llm_base_url", OPENAI_COMPAT_BASE,
    "--force_index_from_scratch", FORCE_INDEX_FROM_SCRATCH,
    "--force_openie_from_scratch", FORCE_OPENIE_FROM_SCRATCH,
]
if SAMPLE_N is not None:
    cmd += ["--sample", str(SAMPLE_N)]

proc = subprocess.Popen(
    cmd,
    cwd="/content/GraphRAG-Benchmark",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

with open(RUN_LOG, "w", encoding="utf-8") as f:
    for line in proc.stdout:
        print(line, end="")
        f.write(line)

rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"HippoRAG2 run failed with exit code {rc}. See: {RUN_LOG}")


RUN_TAG: 20260414_150831
RUN_DIR: /content/GraphRAG-Benchmark/results/runs/20260414_150831
WORKSPACE_DIR: /content/workspaces/hipporag2_workspace
SAMPLE_N: 128
FORCE_INDEX_FROM_SCRATCH: true
FORCE_OPENIE_FROM_SCRATCH: true
Log: /content/GraphRAG-Benchmark/results/runs/20260414_150831/logs/run_hipporag2_medical.log
2026-04-14 15:08:42.806056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776179322.830088   13271 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776179322.838027   13271 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776179322.859579   13271 computation_placer.cc:177] computation placer already registered. Please check link

In [ ]:
# Find produced predictions file (with helpful debug output)
import glob

preds = sorted(glob.glob("/content/GraphRAG-Benchmark/results/hipporag2/*/predictions_*.json"))
print("Found", len(preds), "prediction files")
for p in preds[:5]:
    print(p)

if not preds:
    print("\nNo predictions found.")
    print("Run log:", globals().get("RUN_LOG"))
    try:
        run_log = globals().get("RUN_LOG")
        if run_log:
            !bash -lc "tail -n 120 \"$RUN_LOG\" || true"
    except Exception as e:
        print("Could not tail RUN_LOG:", e)
    !bash -lc "ls -la /content/GraphRAG-Benchmark/results || true"
    !bash -lc "ls -la /content/GraphRAG-Benchmark/results/hipporag2 || true"
    raise AssertionError("No predictions found. Check run log, /content/GraphRAG-Benchmark/hipporag_processing.log and /content/ollama.log")

PRED_FILE = preds[0]
print("Using:", PRED_FILE)


Found 1 prediction files
/content/GraphRAG-Benchmark/results/hipporag2/Medical/predictions_Medical.json
Using: /content/GraphRAG-Benchmark/results/hipporag2/Medical/predictions_Medical.json


In [ ]:
from pathlib import Path
import os, signal

pid_file = Path(f"{LOG_DIR}/generation_eval.pid")
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    print("Killing old PID:", pid)
    try:
        os.kill(pid, signal.SIGTERM)
    except Exception as e:
        print("kill failed:", e)
else:
    print("No existing PID file.")


No existing PID file.


In [ ]:
import os
from huggingface_hub import snapshot_download

os.environ["HF_HOME"] = os.environ.get("HF_HOME", "/content/hf")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("EVAL_EMBED_MODEL_ID(before):", EVAL_EMBED_MODEL_ID)
local_dir = snapshot_download(repo_id=EVAL_EMBED_MODEL_ID, cache_dir=os.environ["HF_HOME"])
EVAL_EMBED_MODEL_ID = local_dir
print("EVAL_EMBED_MODEL_ID(after):", EVAL_EMBED_MODEL_ID)

# bundan sonra HF'ye tekrar gitmesin
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


EVAL_EMBED_MODEL_ID(before): BAAI/bge-large-en-v1.5


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

EVAL_EMBED_MODEL_ID(after): /content/hf/models--BAAI--bge-large-en-v1.5/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09


In [ ]:
import os, requests, json, time

OPENAI_COMPAT_BASE = OPENAI_COMPAT_BASE.replace("localhost", "127.0.0.1")

for k in ["HTTP_PROXY","HTTPS_PROXY","http_proxy","https_proxy"]:
    os.environ.pop(k, None)
os.environ["NO_PROXY"] = "127.0.0.1,localhost"
os.environ["no_proxy"] = "127.0.0.1,localhost"

# hızlı kontrol
print("GET /models:", requests.get(f"{OPENAI_COMPAT_BASE}/models", timeout=5).status_code)

# warmup: modeli yüklet
payload = {
  "model": MODEL_NAME,
  "messages": [{"role":"user","content":"Say OK."}],
  "temperature": 0
}
r = requests.post(f"{OPENAI_COMPAT_BASE}/chat/completions", json=payload, timeout=180)
print("warmup status:", r.status_code)
print("warmup text:", r.text[:200])


GET /models: 401
warmup status: 401
warmup text: {
    "error": {
        "message": "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the passw


In [ ]:
# GEN eval paths
GEN_EVAL_PATH = f"{RUN_DIR}/artifacts/eval_generation.json"
GEN_EVAL_LOG  = f"{LOG_DIR}/generation_eval.log"

# localhost yerine 127.0.0.1 daha stabil
OPENAI_COMPAT_BASE = OPENAI_COMPAT_BASE.replace("localhost", "127.0.0.1")

# proxy varsa local'e karışmasın
import os
for k in ["HTTP_PROXY","HTTPS_PROXY","http_proxy","https_proxy"]:
    os.environ.pop(k, None)
os.environ["NO_PROXY"] = "127.0.0.1,localhost"
os.environ["no_proxy"] = "127.0.0.1,localhost"

# ChatOpenAI key istiyor (local için placeholder yeter)
os.environ.setdefault("OPENAI_API_KEY", "sk-ollama")
os.environ.setdefault("LLM_API_KEY", os.environ["OPENAI_API_KEY"])

print("GEN_EVAL_PATH:", GEN_EVAL_PATH)
print("GEN_EVAL_LOG:", GEN_EVAL_LOG)
print("OPENAI_COMPAT_BASE:", OPENAI_COMPAT_BASE)
print("SAMPLE_N:", SAMPLE_N)

%cd /content/GraphRAG-Benchmark
!bash -lc "set -o pipefail; python -u -m Evaluation.generation_eval \
  --mode API \
  --model \"$MODEL_NAME\" \
  --base_url \"$OPENAI_COMPAT_BASE\" \
  --embedding_model \"$EVAL_EMBED_MODEL_ID\" \
  --data_file \"$PRED_FILE\" \
  --output_file \"$GEN_EVAL_PATH\" \
  --num_samples $SAMPLE_N \
  2>&1 | tee \"$GEN_EVAL_LOG\""


GEN_EVAL_PATH: /content/GraphRAG-Benchmark/results/runs/20260414_150831/artifacts/eval_generation.json
GEN_EVAL_LOG: /content/GraphRAG-Benchmark/results/runs/20260414_150831/logs/generation_eval.log
OPENAI_COMPAT_BASE: https://api.openai.com/v1
SAMPLE_N: 128
/content/GraphRAG-Benchmark
/usr/lib/python3.12/asyncio/base_events.py:1999: UserWarning: Parameters {'frequency_penalty', 'seed', 'top_p', 'presence_penalty'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  handle._run()
/content/GraphRAG-Benchmark/Evaluation/generation_eval.py:170: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = Huggi

In [ ]:
# Prefetch embedding only if it's a HF repo id; if already local path, keep it.
import os
from pathlib import Path

def is_local_path(s: str) -> bool:
    return s.startswith("/") or s.startswith("./") or s.startswith("../") or (":" in s)

print("EVAL_EMBED_MODEL_ID(before):", EVAL_EMBED_MODEL_ID)

if is_local_path(EVAL_EMBED_MODEL_ID) and Path(EVAL_EMBED_MODEL_ID).exists():
    print("Already local. Skipping download.")
else:
    from huggingface_hub import snapshot_download
    os.environ["HF_HOME"] = os.environ.get("HF_HOME", "/content/hf")
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    local_dir = snapshot_download(repo_id=EVAL_EMBED_MODEL_ID, cache_dir=os.environ["HF_HOME"])
    EVAL_EMBED_MODEL_ID = local_dir
    print("Downloaded to:", local_dir)

print("EVAL_EMBED_MODEL_ID(after):", EVAL_EMBED_MODEL_ID)

# Optional: go offline after model is ready
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


EVAL_EMBED_MODEL_ID(before): /content/hf/models--BAAI--bge-large-en-v1.5/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09
Already local. Skipping download.
EVAL_EMBED_MODEL_ID(after): /content/hf/models--BAAI--bge-large-en-v1.5/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09


In [ ]:
import requests, os
print(requests.get("https://api.openai.com/v1/models", headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}, timeout=10).status_code)


200


In [ ]:
# Retrieval evaluation (logged; saved per run) - robust env + unbuffered
%cd /content/GraphRAG-Benchmark
import os, sys, subprocess

RET_EVAL_PATH = f"{RUN_DIR}/artifacts/eval_retrieval.json"
RET_EVAL_LOG = f"{LOG_DIR}/retrieval_eval.log"
print("RET_EVAL_PATH:", RET_EVAL_PATH)
print("RET_EVAL_LOG:", RET_EVAL_LOG)

# Subprocess env: stop TF/Flax imports + force HF offline (since model already local) + unbuffer
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["TRANSFORMERS_NO_TF"] = "1"
env["TRANSFORMERS_NO_FLAX"] = "1"
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
env["HF_HUB_OFFLINE"] = "1"
env["TRANSFORMERS_OFFLINE"] = "1"

# Ensure OpenAI keys exist (for 4o-mini)
env["OPENAI_API_KEY"] = env.get("OPENAI_API_KEY", "")
env["LLM_API_KEY"] = env.get("LLM_API_KEY", env["OPENAI_API_KEY"])

cmd = [
    sys.executable, "-u", "-m", "Evaluation.retrieval_eval",
    "--mode", "API",
    "--model", MODEL_NAME,
    "--base_url", OPENAI_COMPAT_BASE,
    "--embedding_model", EVAL_EMBED_MODEL_ID,  # local path ise OK
    "--data_file", PRED_FILE,
    "--output_file", RET_EVAL_PATH,
]
if SAMPLE_N is not None:
    cmd += ["--num_samples", str(SAMPLE_N)]

with open(RET_EVAL_LOG, "w", encoding="utf-8") as f:
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"retrieval_eval failed with exit code {rc}. See: {RET_EVAL_LOG}")
print("Done:", RET_EVAL_PATH)


/content/GraphRAG-Benchmark
RET_EVAL_PATH: /content/GraphRAG-Benchmark/results/runs/20260414_150831/artifacts/eval_retrieval.json
RET_EVAL_LOG: /content/GraphRAG-Benchmark/results/runs/20260414_150831/logs/retrieval_eval.log
/usr/lib/python3.12/asyncio/tasks.py:303: UserWarning: Parameters {'presence_penalty', 'seed', 'top_p', 'frequency_penalty'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  self.__step_run_and_handle_result(exc)
/content/GraphRAG-Benchmark/Evaluation/retrieval_eval.py:155: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceBgeEmbeddings(model_name=args.embedding

In [ ]:
# Indexing evaluation (graph metrics) for HippoRAG2 (saved per run)
# hipporag stores graph under: base_dir/<corpus_name>/<llm_label>_<embedding_label>/graph.pickle
llm_label = MODEL_NAME.replace('/', "_")
embedding_label = f"Transformers/{EMBED_MODEL_ID}".replace('/', "_")
folder_name = f"{llm_label}_{embedding_label}"
IDX_EVAL_PATH = f"{RUN_DIR}/artifacts/indexing_hipporag2_medical.txt"
IDX_EVAL_LOG = f"{LOG_DIR}/indexing_eval.log"
print("folder_name:", folder_name)
print("IDX_EVAL_PATH:", IDX_EVAL_PATH)
%cd /content/GraphRAG-Benchmark
!bash -lc "set -o pipefail; python -m Evaluation.indexing_eval \
  --framework hipporag2 \
  --base_path /content/workspaces/hipporag2_workspace \
  --folder_name \"$folder_name\" \
  --output \"$IDX_EVAL_PATH\" \
  2>&1 | tee \"$IDX_EVAL_LOG\""
!sed -n '1,80p' "$IDX_EVAL_PATH"


folder_name: gpt-4o-mini_Transformers_BAAI_bge-base-en-v1.5
IDX_EVAL_PATH: /content/GraphRAG-Benchmark/results/runs/20260414_150831/artifacts/indexing_hipporag2_medical.txt
/content/GraphRAG-Benchmark
Calculating indexing graph metrics for hipporag2...
Base path: /content/workspaces/hipporag2_workspace
Folder name: gpt-4o-mini_Transformers_BAAI_bge-base-en-v1.5

Results saved to /content/GraphRAG-Benchmark/results/runs/20260414_150831/artifacts/indexing_hipporag2_medical.txt
Average metrics for hipporag2:
  num_nodes: 9201.0000
  num_edges: 88297.0000
  average_degree: 19.1929
  density: 0.0021
  num_components: 9.0000
  largest_component_size: 9193.0000
  average_clustering_coefficient: 0.4007
  diameter: inf
  average_component_size: 9193.0000
  median_component_size: 9193.0000
  trimmed_mean_component_size: 9193.0000
  geometric_mean_component_size: 9193.0000
  harmonic_mean_component_size: 9193.0000
  num_components_excluding_isolated: 1.0000
  num_components_above_average: 0.0000


In [ ]:
# Summarize results + checkpoint artifacts + zip download (safe copy)
import json, shutil, glob, os
from pathlib import Path

preds = sorted(glob.glob("/content/GraphRAG-Benchmark/results/hipporag2/*/predictions_*.json"))
print("Found", len(preds), "prediction files")
for p in preds[:5]:
    print(p)
assert preds, "No predictions found"

PRED_FILE = preds[0]
print("Using:", PRED_FILE)

# Stage artifacts
art_dir = Path(RUN_DIR) / "artifacts"
art_dir.mkdir(parents=True, exist_ok=True)

# Record environment + config for reproducibility
import platform, subprocess
cfg = {
    "run_tag": RUN_TAG,
    "model_name": MODEL_NAME,
    "embed_model_id": EMBED_MODEL_ID,
    "eval_embed_model_id": EVAL_EMBED_MODEL_ID,
    "sample_n": SAMPLE_N,
    "openai_compat_base": OPENAI_COMPAT_BASE,
    "workspace_dir": WORKSPACE_DIR,
}
(art_dir / "run_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")

versions = {
    "python": platform.python_version(),
}
for pkg in ["torch", "transformers", "sentence_transformers", "openai", "hipporag", "igraph"]:
    try:
        mod = __import__(pkg)
        versions[pkg] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        versions[pkg] = f"unavailable: {e}"
(art_dir / "versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")

try:
    freeze = subprocess.check_output(["pip", "freeze"], text=True)
    (art_dir / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
except Exception as e:
    (art_dir / "pip_freeze_error.txt").write_text(str(e), encoding="utf-8")


def safe_copy(src, dst_dir):
    src = Path(src)
    if not src.exists():
        return False
    dst = Path(dst_dir) / src.name
    try:
        if dst.exists() and src.resolve() == dst.resolve():
            return False  # already there
    except Exception:
        pass
    shutil.copy2(src, dst)
    return True

# copy predictions to a stable name
shutil.copy2(PRED_FILE, art_dir / "predictions.json")

# copy eval jsons + logs (skip if already in artifacts)
for p in [GEN_EVAL_PATH, RET_EVAL_PATH, IDX_EVAL_PATH]:
    safe_copy(p, art_dir)

for p in glob.glob(f"{LOG_DIR}/*.log"):
    safe_copy(p, art_dir)

print("Artifacts:", str(art_dir))

# Compact summary
def load_json(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else None

gen = load_json(GEN_EVAL_PATH)
ret = load_json(RET_EVAL_PATH)

print("\n=== SUMMARY ===")
print("model:", MODEL_NAME)
print("sample:", "ALL" if SAMPLE_N is None else SAMPLE_N)

if gen is not None:
    print("\nGeneration:")
    for k, v in gen.items():
        if isinstance(v, dict) and "average_scores" in v:
            print(f"- {k}: {v['average_scores']}")
        else:
            print(f"- {k}: {v}")

if ret is not None:
    print("\nRetrieval:")
    for k, v in ret.items():
        print(f"- {k}: {v}")

# Zip (include run dir + the raw hipporag2 results folder)
ZIP_PATH = f"/content/hipporag2_run_{RUN_TAG}.zip"
!bash -lc "cd /content && zip -r \"{ZIP_PATH}\" \"GraphRAG-Benchmark/results/runs/{RUN_TAG}\" GraphRAG-Benchmark/results/hipporag2 >/dev/null"
print("ZIP:", ZIP_PATH)

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as e:
    print("Download skipped (not in Colab):", e)


Found 1 prediction files
/content/GraphRAG-Benchmark/results/hipporag2/Medical/predictions_Medical.json
Using: /content/GraphRAG-Benchmark/results/hipporag2/Medical/predictions_Medical.json
Artifacts: /content/GraphRAG-Benchmark/results/runs/20260414_150831/artifacts

=== SUMMARY ===
model: gpt-4o-mini
sample: 128

Generation:
- Fact Retrieval: {'rouge_score': 0.3198504688719257, 'answer_correctness': 0.655963617525994}

Retrieval:
- Fact Retrieval: {'context_relevancy': 0.85546875, 'evidence_recall': 0.9035528273809523}
ZIP: /content/hipporag2_run_20260414_150831.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>